## 📥 Início de Nova Análise – Ingestão de Arquivo

Leitura de um arquivo em formato Excel (.xlsx) para carregamento em DataFrame utilizando `pandas`.

Validações realizadas:

- Confirmação de carregamento do arquivo
- Verificação da quantidade de linhas e colunas (`df.shape`)
- Visualização inicial do dataset (`df.head()`)

Objetivo: disponibilizar os dados para análise exploratória e etapas posteriores de tratamento e transformação.

In [1]:
import pandas as pd

path = r"C:\AZURE_DEVOS_REPOS\RSM_DATALAKE_JOURNAL_ENTRIES\033-DTLK-Data Analytics_Viação Normandy do Triângulo Ltda_31.12.2025\bronze\razao\Razão Completo (1).xlsx"

df = pd.read_excel(path)

print("✅ Arquivo carregado com sucesso")
print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

df.head(1)


✅ Arquivo carregado com sucesso
Linhas: 215450
Colunas: 9


,Conta,Conta.1,Descrição,DATA,HISTORICO,C/PARTIDA,DEBITO,CREDITO,MOVIMENTO
0,1.1.1,1.1.1.01.01.001.00001,NUMERARIO EM CAIXA,2025-01-02,TRANF BANCARIA ORIGEM: CX3-0000-00000000,NaN,0.0,-8000.0,-8000.0


## 📋 Verificação de Colunas

Exibição da lista completa de colunas do DataFrame utilizando:

- `df.columns.tolist()`

Objetivo: validar o schema do dataset e identificar possíveis inconsistências nos nomes das colunas.

In [3]:
print(df.columns.tolist())

['Conta', 'Conta.1', 'Descrição', 'DATA', 'HISTORICO', 'C/PARTIDA', 'DEBITO', 'CREDITO', 'MOVIMENTO']


## 🔄 Padronização de Nomes de Colunas

Renomeação de colunas para padrão técnico (uppercase e nomenclatura padronizada), visando:

- Uniformização do schema
- Facilitar aplicação de regras contábeis
- Padronizar integrações futuras

Validações realizadas:

- Exibição da nova lista de colunas (`.columns.tolist()`)
- Visualização preliminar do dataset (`.head(1)`)

In [4]:
df_razao_renomeado = df.rename(columns={
    "Conta.1": "CONTA",
    "C/PARTIDA": "CONTRAPARTIDA"
})
print(df_razao_renomeado.columns.tolist())
df_razao_renomeado.head(1)


['Conta', 'CONTA', 'Descrição', 'DATA', 'HISTORICO', 'CONTRAPARTIDA', 'DEBITO', 'CREDITO', 'MOVIMENTO']


,Conta,CONTA,Descrição,DATA,HISTORICO,CONTRAPARTIDA,DEBITO,CREDITO,MOVIMENTO
0,1.1.1,1.1.1.01.01.001.00001,NUMERARIO EM CAIXA,2025-01-02,TRANF BANCARIA ORIGEM: CX3-0000-00000000,NaN,0.0,-8000.0,-8000.0


## 📊 Inspeção de Tipos de Dados

Geração de DataFrame auxiliar contendo:

- Nome das colunas
- Tipo de dado correspondente no pandas

Objetivo: validar o schema após a padronização e garantir consistência para etapas de transformação.

In [5]:
df_info = pd.DataFrame({
    "coluna": df_razao_renomeado.columns,
    "tipo_pandas": df_razao_renomeado.dtypes.astype(str)
})

df_info


,coluna,tipo_pandas
Conta,Conta,object
CONTA,CONTA,object
Descrição,Descrição,object
DATA,DATA,datetime64[ns]
HISTORICO,HISTORICO,object
CONTRAPARTIDA,CONTRAPARTIDA,object
DEBITO,DEBITO,float64
CREDITO,CREDITO,float64
MOVIMENTO,MOVIMENTO,float64


## 🔍 Verificação do Último Registro

Visualização da última linha do DataFrame utilizando:

- `df.tail(1)`

Objetivo: validar a integridade do dataset e confirmar a correta leitura até o final do arquivo.

In [6]:
df_razao_renomeado.tail(1)


,Conta,CONTA,Descrição,DATA,HISTORICO,CONTRAPARTIDA,DEBITO,CREDITO,MOVIMENTO
215449,2.3.3,7.7.0.10.02.012,PREJUIZOS DO MES DE DEZEMBRO,2025-12-31,APURACAO RESULTADO 122025,2.4.5.01.01.002.00001,0.0,-2112880.86,-2112880.86


## 🧹 Limpeza e Normalização de Texto

Aplicação de tratamento robusto na coluna `HISTORICO`, gerando nova coluna `Historico_LIMPO` com:

- Remoção de caracteres especiais (`/`, `-`, símbolos)
- Normalização de acentuação (NFKD → ASCII)
- Padronização de espaços
- Conversão para caixa alta (uppercase)

Objetivo: padronizar descrições textuais para garantir consistência em análises, filtros e aplicação de regras automatizadas.

Validação realizada com visualização comparativa entre `HISTORICO` original e versão tratada.

In [7]:
# =========================
# LIMPEZA ROBUSTA
# =========================
df_razao_renomeado["Histórico_LIMPO"] = (
    df_razao_renomeado["HISTORICO"]
        .astype(str)
        .str.replace(r"[\/\-]+", " ", regex=True)      # remove / e -
        .str.normalize("NFKD")                         # normaliza acentos
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.replace(r"[^A-Za-z0-9 ]+", " ", regex=True)  # remove símbolos
        .str.replace(r"\s+", " ", regex=True)          # normaliza espaços
        .str.strip()
        .str.upper()
)

# =========================
# DEPOIS DA LIMPEZA
# =========================
print("\n🟢 DEPOIS DA LIMPEZA:")
display(df_razao_renomeado[["HISTORICO", "Histórico_LIMPO"]].head(10))


🟢 DEPOIS DA LIMPEZA:


,HISTORICO,Histórico_LIMPO
0,TRANF BANCARIA ORIGEM: CX3-0000-00000000,TRANF BANCARIA ORIGEM CX3 0000 00000000
1,00,00
2,TRANF BANCARIA DESTINO: CX3-0000-0000000,TRANF BANCARIA DESTINO CX3 0000 0000000
3,000,000
4,TRANF BANCARIA ORIGEM: CX3-0000-00000000,TRANF BANCARIA ORIGEM CX3 0000 00000000
5,00,00
6,TRANF BANCARIA DESTINO: CX3-0000-0000000,TRANF BANCARIA DESTINO CX3 0000 0000000
7,000,000
8,TRANF BANCARIA ORIGEM: CX3-0000-00000000,TRANF BANCARIA ORIGEM CX3 0000 00000000
9,00,00


## 🔄 Substituição da Coluna de Histórico

Procedimento para substituir a coluna original `HISTORICO` pela versão tratada:

- Remoção de espaços extras nos nomes das colunas
- Exclusão da coluna `HISTORICO` original (se existir)
- Renomeação da coluna limpa (`Historico_LIMPO` ou `Histórico_LIMPO`) para `HISTORICO`

Objetivo: manter o schema padronizado, garantindo que apenas a versão normalizada da coluna permaneça no dataset.

In [8]:
df_razao_renomeado.columns = df_razao_renomeado.columns.str.strip()

# 1) remove a coluna antiga, se existir
if "HISTORICO" in df_razao_renomeado.columns:
    df_razao_renomeado = df_razao_renomeado.drop(columns=["HISTORICO"])

# 2) renomeia a limpa (tenta com acento e sem acento)
if "Histórico_LIMPO" in df_razao_renomeado.columns:
    df_razao_renomeado = df_razao_renomeado.rename(columns={"Histórico_LIMPO": "HISTORICO"})
elif "Historico_LIMPO" in df_consolidado.columns:
    df_razao_renomeado = df_razao_renomeado.rename(columns={"Historico_LIMPO": "HISTORICO"})
else:
    raise KeyError("Não achei 'Histórico_LIMPO' nem 'Historico_LIMPO'. Rode o print(repr(c)) para ver o nome exato.")


## 📋 Validação de Schema e Preview

- Exibição da lista atual de colunas (`df.columns.tolist()`)
- Visualização dos três primeiros registros (`df.head(3)`)

Objetivo: confirmar a estrutura final do DataFrame após as transformações e validar a integridade inicial dos dados.

In [9]:
print(df_razao_renomeado.columns.tolist())
df_razao_renomeado.head(1)


['Conta', 'CONTA', 'Descrição', 'DATA', 'CONTRAPARTIDA', 'DEBITO', 'CREDITO', 'MOVIMENTO', 'HISTORICO']


,Conta,CONTA,Descrição,DATA,CONTRAPARTIDA,DEBITO,CREDITO,MOVIMENTO,HISTORICO
0,1.1.1,1.1.1.01.01.001.00001,NUMERARIO EM CAIXA,2025-01-02,NaN,0.0,-8000.0,-8000.0,TRANF BANCARIA ORIGEM CX3 0000 00000000


## 🧹 Padronização de Campos Numéricos

Aplicação de limpeza nas colunas `CONTA` e `CONTRAPARTIDA`:

- `CONTA`: remoção de pontos e hífens
- `CONTRAPARTIDA`: remoção de caracteres não alfanuméricos

Objetivo: padronizar identificadores contábeis, evitando inconsistências em junções, agrupamentos e regras de validação.

Visualização realizada com `df.head()` para conferência.

In [22]:
# Limpeza da CONTA
df_razao_renomeado["CONTA"] = (
    df_razao_renomeado["CONTA"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace("-", "", regex=False)
)

# Limpeza da Contrapartida
df_razao_renomeado["CONTRAPARTIDA"] = (
    df_razao_renomeado["CONTRAPARTIDA"]
    .astype(str)
    .str.replace(r"[^a-zA-Z0-9]", "", regex=True)
)


df_razao_renomeado.head(1)


,Conta,CONTA,Descrição,DATA,CONTRAPARTIDA,DEBITO,CREDITO,MOVIMENTO,HISTORICO
0,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-02,nan,0.0,-8000.0,-8000.0,TRANF BANCARIA ORIGEM CX3 0000 00000000


## 🔎 Extração de Componentes da Contrapartida

Aplicação de regex para segmentar a coluna `CONTRAPARTIDA` em dois campos:

- `CONTA_CONTRAPARTIDA`: extração da parte numérica inicial
- `DESCRICAO_CONTRAPARTIDA`: extração da parte textual subsequente

Objetivo: estruturar o campo de contrapartida em componentes distintos, facilitando análises, cruzamentos com plano de contas e validações contábeis.

NAO UTIZADO

In [12]:
import re

# Extrair parte numérica (início da string)
df_razao_renomeado["CONTA_CONTRAPARTIDA"] = (
    df_razao_renomeado["CONTRAPARTIDA"]
    .str.extract(r'^(\d+)')
)

# Extrair parte textual (restante da string)
df_razao_renomeado["DESCRICAO_CONTRAPARTIDA"] = (
    df_razao_renomeado["CONTRAPARTIDA"]
    .str.extract(r'^\d+(.*)')
)


## 📋 Validação de Schema e Preview

- Exibição da lista atual de colunas (`df.columns.tolist()`)
- Visualização dos três primeiros registros (`df.head(3)`)

Objetivo: confirmar a estrutura final do DataFrame após as transformações e validar a integridade inicial dos dados.

In [23]:
print(df_razao_renomeado.columns.tolist())
df_razao_renomeado.head(1)


['Conta', 'CONTA', 'Descrição', 'DATA', 'CONTRAPARTIDA', 'DEBITO', 'CREDITO', 'MOVIMENTO', 'HISTORICO']


,Conta,CONTA,Descrição,DATA,CONTRAPARTIDA,DEBITO,CREDITO,MOVIMENTO,HISTORICO
0,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-02,nan,0.0,-8000.0,-8000.0,TRANF BANCARIA ORIGEM CX3 0000 00000000


In [24]:
print(df_razao_renomeado.columns.tolist())

['Conta', 'CONTA', 'Descrição', 'DATA', 'CONTRAPARTIDA', 'DEBITO', 'CREDITO', 'MOVIMENTO', 'HISTORICO']


## 📑 Reordenação de Colunas

Definição de nova ordem lógica para as colunas do DataFrame, organizando:

- Informações de identificação (DATA, Lote, CONTA)
- Dados contábeis (DEBITO, CREDITO, MOVIMENTO, Saldo)
- Histórico padronizado
- Componentes estruturados da contrapartida

Objetivo: padronizar a estrutura do dataset para facilitar leitura, auditoria e exportação.

Validação realizada com `df.head(1)`.

In [25]:
nova_ordem = ['Conta', 'CONTA', 'Descrição', 'DATA', 'CONTRAPARTIDA', 'DEBITO', 'CREDITO', 'MOVIMENTO', 'HISTORICO']

df_razao_renomeado = df_razao_renomeado[nova_ordem]

df_razao_renomeado.head(1)


,Conta,CONTA,Descrição,DATA,CONTRAPARTIDA,DEBITO,CREDITO,MOVIMENTO,HISTORICO
0,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-02,nan,0.0,-8000.0,-8000.0,TRANF BANCARIA ORIGEM CX3 0000 00000000


## 📥 Ingestão – Arquivo de Plano de Contas

Leitura de arquivo Excel (.xlsx) para carregamento em DataFrame utilizando `pandas`.

Validações realizadas:

- Confirmação de carregamento do arquivo
- Verificação da dimensão do dataset (`shape`)
- Visualização preliminar dos dados (`head()`)

Objetivo: disponibilizar o plano de contas para cruzamento com os lançamentos contábeis e aplicação de regras de classificação.

In [26]:
import pandas as pd

path = r"C:\AZURE_DEVOS_REPOS\RSM_DATALAKE_JOURNAL_ENTRIES\033-DTLK-Data Analytics_Viação Normandy do Triângulo Ltda_31.12.2025\bronze\balancete\balancete.xlsx"

df_bal = pd.read_excel(path)

print("✅ Arquivo carregado com sucesso")
print(f"Linhas: {df_bal.shape[0]}")
print(f"Colunas: {df_bal.shape[1]}")

df_bal.head(1)


✅ Arquivo carregado com sucesso
Linhas: 652
Colunas: 6


,Empresa,N° da conta,Descrição da conta,Analítica | Sintética,Ativo | Passivo | PL | Resultado,Grupo
0,Normandy,1.1.1.01.01.001.00001,NUMERARIO EM CAIXA,Sintética,Ativo,Caixa e equivalentes de caixa


## 🔄 Padronização de Nomes de Colunas

Renomeação de colunas para manter um padrão uniforme no DataFrame.

Objetivo: garantir consistência estrutural e facilitar integrações, análises e cruzamentos com outros datasets.

Validação realizada com exibição da lista de colunas e visualização inicial dos dados.

In [27]:
df_bal = df_bal.rename(columns={
    "N° da conta": "CONTA",
    "Grupo": "GRUPO",
    "Ativo | Passivo | PL | Resultado":"CTP"
})
print(df_bal.columns.tolist())
df_bal.head()


['Empresa', 'CONTA', 'Descrição da conta', 'Analítica | Sintética', 'CTP', 'GRUPO']


,Empresa,CONTA,Descrição da conta,Analítica | Sintética,CTP,GRUPO
0,Normandy,1.1.1.01.01.001.00001,NUMERARIO EM CAIXA,Sintética,Ativo,Caixa e equivalentes de caixa
1,Normandy,1.1.1.01.01.001.00005,FUNDO FIXO,Sintética,Ativo,Caixa e equivalentes de caixa
2,Normandy,1.1.1.01.01.001.00008,FUNDO FIXO ALELO NORMANDY,Sintética,Ativo,Caixa e equivalentes de caixa
3,Normandy,1.1.1.01.03.001.00320,BCO GUANABARA AG 001 C/C 03627-3,Sintética,Ativo,Caixa e equivalentes de caixa
4,Normandy,1.1.1.01.03.001.00322,BCO BRADESCO AG 3370 CTA 44,Sintética,Ativo,Caixa e equivalentes de caixa


## 🧹 Padronização de Campo Identificador

Aplicação de limpeza na coluna de identificação (`CONTA`), removendo caracteres específicos e garantindo formato consistente em string.

Objetivo: padronizar o campo para evitar inconsistências em junções, comparações e validações futuras.

Validação realizada com visualização do DataFrame (`head()`).

In [28]:
# Limpeza da CONTA
df_bal["CONTA"] = (
    df_bal["CONTA"]
    .astype(str)
    .str.replace(".", "", regex=False) 
)

df_bal.head()


,Empresa,CONTA,Descrição da conta,Analítica | Sintética,CTP,GRUPO
0,Normandy,111010100100001,NUMERARIO EM CAIXA,Sintética,Ativo,Caixa e equivalentes de caixa
1,Normandy,111010100100005,FUNDO FIXO,Sintética,Ativo,Caixa e equivalentes de caixa
2,Normandy,111010100100008,FUNDO FIXO ALELO NORMANDY,Sintética,Ativo,Caixa e equivalentes de caixa
3,Normandy,111010300100320,BCO GUANABARA AG 001 C/C 03627-3,Sintética,Ativo,Caixa e equivalentes de caixa
4,Normandy,111010300100322,BCO BRADESCO AG 3370 CTA 44,Sintética,Ativo,Caixa e equivalentes de caixa


In [29]:
print(df_bal.columns.tolist())
df_bal.head(10)


['Empresa', 'CONTA', 'Descrição da conta', 'Analítica | Sintética', 'CTP', 'GRUPO']


,Empresa,CONTA,Descrição da conta,Analítica | Sintética,CTP,GRUPO
0,Normandy,111010100100001,NUMERARIO EM CAIXA,Sintética,Ativo,Caixa e equivalentes de caixa
1,Normandy,111010100100005,FUNDO FIXO,Sintética,Ativo,Caixa e equivalentes de caixa
2,Normandy,111010100100008,FUNDO FIXO ALELO NORMANDY,Sintética,Ativo,Caixa e equivalentes de caixa
3,Normandy,111010300100320,BCO GUANABARA AG 001 C/C 03627-3,Sintética,Ativo,Caixa e equivalentes de caixa
4,Normandy,111010300100322,BCO BRADESCO AG 3370 CTA 44,Sintética,Ativo,Caixa e equivalentes de caixa
5,Normandy,111010300100323,BCO BRADESCO AG 3370 CTA 47,Sintética,Ativo,Caixa e equivalentes de caixa
6,Normandy,111010300100324,BCO ITAU AG 584 CTA 56330,Sintética,Ativo,Caixa e equivalentes de caixa
7,Normandy,111010300100325,BCO SANTANDER AG 3403 13066285-9,Sintética,Ativo,Caixa e equivalentes de caixa
8,Normandy,111010300100326,BCO SANTANDER AG 3003 CTA 130827551,Sintética,Ativo,Caixa e equivalentes de caixa
9,Normandy,111010300100327,BCO SANTANDER AG 3003 CTA 13058947,Sintética,Ativo,Caixa e equivalentes de caixa


## 💾 Persistência na Camada Silver

Exportação do DataFrame tratado para a camada **Silver** do Data Lake.

Procedimentos realizados:

- Definição do diretório de destino
- Criação da pasta (caso não exista)
- Geração do arquivo `.xlsx`
- Salvamento do dataset sem índice

Objetivo: armazenar a versão tratada e padronizada do dataset para uso em etapas analíticas e cruzamentos posteriores.

In [17]:
import os
# 📂 Caminho destino (Silver)
pasta_silver = r"C:\AZURE_DEVOS_REPOS\RSM_DATALAKE_JOURNAL_ENTRIES\033-DTLK-Data Analytics_Viação Normandy do Triângulo Ltda_31.12.2025\silver"

# Criar pasta se não existir
os.makedirs(pasta_silver, exist_ok=True)

# Nome do arquivo de saída
arquivo_saida = os.path.join(pasta_silver, "balancete_NORMANDY_silver.xlsx")

# 💾 Exportar para XLSX
df_bal.to_excel(arquivo_saida, index=False)

print(f"✅ Arquivo exportado com sucesso para:\n{arquivo_saida}")

✅ Arquivo exportado com sucesso para:
C:\AZURE_DEVOS_REPOS\RSM_DATALAKE_JOURNAL_ENTRIES\033-DTLK-Data Analytics_Viação Normandy do Triângulo Ltda_31.12.2025\silver\balancete_NORMANDY_silver.xlsx


## 🔎 Cruzamento de DataFrames

Será realizado um cruzamento entre dois DataFrames para complementar informações.

Objetivo: enriquecer os dados por meio da associação de campos em comum, preparando o dataset para análises e validações posteriores.

BALANCETE


## 📋 Verificação de Estrutura

- Exibição da lista de colunas do DataFrame
- Visualização do primeiro registro (`head(1)`)

Objetivo: validar a estrutura e conferir a organização inicial dos dados.

In [30]:
print(df_bal.columns.tolist())
df_bal.head(1)
 

['Empresa', 'CONTA', 'Descrição da conta', 'Analítica | Sintética', 'CTP', 'GRUPO']


,Empresa,CONTA,Descrição da conta,Analítica | Sintética,CTP,GRUPO
0,Normandy,111010100100001,NUMERARIO EM CAIXA,Sintética,Ativo,Caixa e equivalentes de caixa


## 📊 Inspeção de Schema

Criação de um DataFrame auxiliar contendo:

- Nome das colunas
- Tipo de dado correspondente no pandas

Objetivo: verificar a estrutura do dataset e validar os tipos antes de transformações ou cruzamentos.

In [31]:
df_info = pd.DataFrame({
    "coluna": df_bal.columns,
    "tipo_pandas": df_bal.dtypes.astype(str)
})

df_info

,coluna,tipo_pandas
Empresa,Empresa,object
CONTA,CONTA,object
Descrição da conta,Descrição da conta,object
Analítica | Sintética,Analítica | Sintética,object
CTP,CTP,object
GRUPO,GRUPO,object


RAZAO

## 📋 Verificação de Estrutura

- Exibição da lista de colunas do DataFrame
- Visualização do primeiro registro (`head(1)`)

Objetivo: validar a estrutura e conferir a organização inicial dos dados.

In [32]:
print(df_razao_renomeado.columns.tolist())
df_razao_renomeado.head(1)

['Conta', 'CONTA', 'Descrição', 'DATA', 'CONTRAPARTIDA', 'DEBITO', 'CREDITO', 'MOVIMENTO', 'HISTORICO']


,Conta,CONTA,Descrição,DATA,CONTRAPARTIDA,DEBITO,CREDITO,MOVIMENTO,HISTORICO
0,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-02,nan,0.0,-8000.0,-8000.0,TRANF BANCARIA ORIGEM CX3 0000 00000000


## 📊 Inspeção de Schema

Criação de um DataFrame auxiliar contendo:

- Nome das colunas
- Tipo de dado correspondente no pandas

Objetivo: verificar a estrutura do dataset e validar os tipos antes de transformações ou cruzamentos.

In [33]:
df_info = pd.DataFrame({
    "coluna": df_razao_renomeado.columns,
    "tipo_pandas": df_razao_renomeado.dtypes.astype(str)
})

df_info

,coluna,tipo_pandas
Conta,Conta,object
CONTA,CONTA,object
Descrição,Descrição,object
DATA,DATA,datetime64[ns]
CONTRAPARTIDA,CONTRAPARTIDA,object
DEBITO,DEBITO,float64
CREDITO,CREDITO,float64
MOVIMENTO,MOVIMENTO,float64
HISTORICO,HISTORICO,object


## 🔄 Tratamento de Valores Nulos

Substituição do valor textual `"nan"` por `None` na coluna `CONTRAPARTIDA`.

Objetivo: corrigir inconsistências de tipagem e garantir que valores ausentes sejam tratados corretamente como nulos.

Validação realizada com visualização do DataFrame (`head(1)`).

In [25]:
df_razao_renomeado["CONTRAPARTIDA"] = (
    df_razao_renomeado["CONTRAPARTIDA"]
    .replace("nan", None)
)
df_razao_renomeado.head(1)


,CONTA,DescricaoConta,DATA,Unnamed: 3,Lote/Sub/Doc/Linha,CONTRAPARTIDA,FilialOrigem,CentroCusto,ItemConta,CodClasseValor,DEBITO,CREDITO,SaldoAtual,MOVIMENTO,HISTORICO,CONTA_CONTRAPARTIDA,DESCRICAO_CONTRAPARTIDA
0,11110101,CAIXA PEQUENO,2025-07-10,7,FIN001001000017001,11240501,1.0,NaN,NaN,NaN,7694.69,0.0,7694.69,7694.69,DEV ADTO DE FORN 0 ALLOG ASSE,11240501,


## 📑 Seleção e Redução de Colunas

Criação de um DataFrame reduzido contendo apenas colunas relevantes e remoção de duplicidades.

Objetivo: gerar uma base simplificada para cruzamentos e classificações, mantendo apenas informações essenciais.

In [34]:
df_plano_reduzido_BALANCETE_GRUPO = df_bal[["CONTA", "GRUPO"]].drop_duplicates()

df_plano_reduzido_BALANCETE_GRUPO.head(1)




,CONTA,GRUPO
0,111010100100001,Caixa e equivalentes de caixa


## 🔎 Validação Estrutural e Preview

Exibição:

- Lista de colunas de ambos os DataFrames
- Visualização do primeiro registro de cada base

Objetivo: conferir estrutura, alinhamento de campos e consistência inicial antes de realizar integrações ou cruzamentos.

In [35]:
print(df_bal.columns.tolist())
print(df_razao_renomeado.columns.tolist())
print(df_bal.head(1))
print(df_razao_renomeado.head(1))

['Empresa', 'CONTA', 'Descrição da conta', 'Analítica | Sintética', 'CTP', 'GRUPO']
['Conta', 'CONTA', 'Descrição', 'DATA', 'CONTRAPARTIDA', 'DEBITO', 'CREDITO', 'MOVIMENTO', 'HISTORICO']
    Empresa            CONTA  Descrição da conta Analítica | Sintética    CTP  \
0  Normandy  111010100100001  NUMERARIO EM CAIXA             Sintética  Ativo   

                           GRUPO  
0  Caixa e equivalentes de caixa  
   Conta            CONTA           Descrição       DATA CONTRAPARTIDA  \
0  1.1.1  111010100100001  NUMERARIO EM CAIXA 2025-01-02           nan   

   DEBITO  CREDITO  MOVIMENTO                                HISTORICO  
0     0.0  -8000.0    -8000.0  TRANF BANCARIA ORIGEM CX3 0000 00000000  


## 🔎 Aplicação de Cruzamento (Mapeamento)

Procedimento realizado:

- Criação de cópia do DataFrame principal
- Padronização do tipo da coluna chave (`CONTA`)
- Construção de mapas de referência a partir de outra base
- Aplicação de mapeamento para inclusão de novas colunas classificatórias

Objetivo: enriquecer o dataset principal com informações complementares por meio de associação baseada em chave comum.

Validação realizada com visualização do resultado inicial.

In [36]:
# 🔹 1. Criar cópia PRIMEIRO
df_razao_procv = df_razao_renomeado.copy()

# 🔹 2. Garantir mesmo tipo
df_razao_procv["CONTA"] = df_razao_procv["CONTA"].astype(str).str.strip()
df_bal["CONTA"] = df_bal["CONTA"].astype(str).str.strip()

# 🔹 3. Criar mapa CONTA → GRUPO
mapa_grupo = df_bal.drop_duplicates("CONTA").set_index("CONTA")["GRUPO"]

# 🔹 4. Criar mapa CONTA → CTP (ajuste o nome da coluna se for diferente)
mapa_ctp = df_bal.drop_duplicates("CONTA").set_index("CONTA")["CTP"]

# 🔹 5. Aplicar PROCV
df_razao_procv["GRUPO_CONTA"] = df_razao_procv["CONTA"].map(mapa_grupo)
df_razao_procv["CLASS_CONTA"] = df_razao_procv["CONTA"].map(mapa_ctp)

# 🔹 6. Conferir
print(df_razao_procv.head(1))


   Conta            CONTA           Descrição       DATA CONTRAPARTIDA  \
0  1.1.1  111010100100001  NUMERARIO EM CAIXA 2025-01-02           nan   

   DEBITO  CREDITO  MOVIMENTO                                HISTORICO  \
0     0.0  -8000.0    -8000.0  TRANF BANCARIA ORIGEM CX3 0000 00000000   

                     GRUPO_CONTA CLASS_CONTA  
0  Caixa e equivalentes de caixa       Ativo  


## 🔎 Enriquecimento via Chave Secundária

Procedimento realizado:

- Criação de novo DataFrame para preservar a base original
- Padronização da coluna chave secundária
- Construção de mapas de referência
- Aplicação de mapeamento para inclusão de classificações da contrapartida

Objetivo: complementar o dataset com informações associadas à chave secundária, ampliando a capacidade de análise e validação contábil.

Validação realizada com visualização preliminar dos registros.

In [37]:
# ✅ Trabalhar sempre no mesmo DF
df_razao_final = df_razao_procv.copy()

# ─────────────────────────────
# 🔹 1. Normalizar chaves no razão
# ─────────────────────────────
df_razao_final["CONTA_CHAVE"] = (
    df_razao_final["CONTA"]
    .fillna("")
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

df_razao_final["CONTRAPARTIDA_CHAVE"] = (
    df_razao_final["CONTRAPARTIDA"]
    .fillna("")
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

# ─────────────────────────────
# 🔹 2. Normalizar chave no balancete
# ─────────────────────────────
df_bal["CONTA_CHAVE"] = (
    df_bal["CONTA"]
    .fillna("")
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.strip()
)

# ─────────────────────────────
# 🔹 3. Criar mapas
# ⚠️ CONFIRA O NOME DAS COLUNAS AQUI
# ─────────────────────────────
print("Colunas df_bal:", df_bal.columns)

mapa_grupo = (
    df_bal
    .drop_duplicates("CONTA_CHAVE")
    .set_index("CONTA_CHAVE")["GRUPO"]  # ajuste se o nome for diferente
)

mapa_class = (
    df_bal
    .drop_duplicates("CONTA_CHAVE")
    .set_index("CONTA_CHAVE")["CTP"]  # ajuste se o nome for diferente
)

# ─────────────────────────────
# 🔹 4. Aplicar PROCV
# ─────────────────────────────
df_razao_final["GRUPO_CONTA"] = df_razao_final["CONTA_CHAVE"].map(mapa_grupo)
df_razao_final["CLASS_CONTA"] = df_razao_final["CONTA_CHAVE"].map(mapa_class)

df_razao_final["GRUPO_CTP"] = df_razao_final["CONTRAPARTIDA_CHAVE"].map(mapa_grupo)
df_razao_final["CLASS_CTP"] = df_razao_final["CONTRAPARTIDA_CHAVE"].map(mapa_class)

# ─────────────────────────────
# 🔍 Diagnóstico real
# ─────────────────────────────
print("GRUPO_CONTA vazios:", df_razao_final["GRUPO_CONTA"].isna().sum())
print("CLASS_CONTA vazios:", df_razao_final["CLASS_CONTA"].isna().sum())
print("GRUPO_CTP vazios:", df_razao_final["GRUPO_CTP"].isna().sum())
print("CLASS_CTP vazios:", df_razao_final["CLASS_CTP"].isna().sum())

print(df_razao_final.head())

Colunas df_bal: Index(['Empresa', 'CONTA', 'Descrição da conta', 'Analítica | Sintética',
       'CTP', 'GRUPO', 'CONTA_CHAVE'],
      dtype='object')
GRUPO_CONTA vazios: 0
CLASS_CONTA vazios: 0
GRUPO_CTP vazios: 121778
CLASS_CTP vazios: 665
   Conta            CONTA           Descrição       DATA CONTRAPARTIDA  \
0  1.1.1  111010100100001  NUMERARIO EM CAIXA 2025-01-02           nan   
1  1.1.1  111010100100001  NUMERARIO EM CAIXA 2025-01-02           nan   
2  1.1.1  111010100100001  NUMERARIO EM CAIXA 2025-01-03           nan   
3  1.1.1  111010100100001  NUMERARIO EM CAIXA 2025-01-03           nan   
4  1.1.1  111010100100001  NUMERARIO EM CAIXA 2025-01-07           nan   

   DEBITO  CREDITO  MOVIMENTO                                HISTORICO  \
0     0.0  -8000.0    -8000.0  TRANF BANCARIA ORIGEM CX3 0000 00000000   
1     NaN      NaN        NaN                                       00   
2  8000.0      0.0     8000.0  TRANF BANCARIA DESTINO CX3 0000 0000000   
3     NaN      Na

In [38]:
print(df_razao_final.columns.tolist())
df_razao_final.head(50)

['Conta', 'CONTA', 'Descrição', 'DATA', 'CONTRAPARTIDA', 'DEBITO', 'CREDITO', 'MOVIMENTO', 'HISTORICO', 'GRUPO_CONTA', 'CLASS_CONTA', 'CONTA_CHAVE', 'CONTRAPARTIDA_CHAVE', 'GRUPO_CTP', 'CLASS_CTP']


,Conta,CONTA,Descrição,DATA,CONTRAPARTIDA,DEBITO,CREDITO,MOVIMENTO,HISTORICO,GRUPO_CONTA,CLASS_CONTA,CONTA_CHAVE,CONTRAPARTIDA_CHAVE,GRUPO_CTP,CLASS_CTP
0,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-02,nan,0.0,-8000.0,-8000.0,TRANF BANCARIA ORIGEM CX3 0000 00000000,Caixa e equivalentes de caixa,Ativo,111010100100001,nan,NaN,Resultado
1,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-02,nan,NaN,NaN,NaN,00,Caixa e equivalentes de caixa,Ativo,111010100100001,nan,NaN,Resultado
2,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-03,nan,8000.0,0.0,8000.0,TRANF BANCARIA DESTINO CX3 0000 0000000,Caixa e equivalentes de caixa,Ativo,111010100100001,nan,NaN,Resultado
3,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-03,nan,NaN,NaN,NaN,000,Caixa e equivalentes de caixa,Ativo,111010100100001,nan,NaN,Resultado
4,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-07,nan,0.0,-8000.0,-8000.0,TRANF BANCARIA ORIGEM CX3 0000 00000000,Caixa e equivalentes de caixa,Ativo,111010100100001,nan,NaN,Resultado
5,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-07,nan,NaN,NaN,NaN,00,Caixa e equivalentes de caixa,Ativo,111010100100001,nan,NaN,Resultado
6,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-10,nan,8000.0,0.0,8000.0,TRANF BANCARIA DESTINO CX3 0000 0000000,Caixa e equivalentes de caixa,Ativo,111010100100001,nan,NaN,Resultado
7,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-10,nan,NaN,NaN,NaN,000,Caixa e equivalentes de caixa,Ativo,111010100100001,nan,NaN,Resultado
8,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-14,nan,0.0,-8000.0,-8000.0,TRANF BANCARIA ORIGEM CX3 0000 00000000,Caixa e equivalentes de caixa,Ativo,111010100100001,nan,NaN,Resultado
9,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-14,nan,NaN,NaN,NaN,00,Caixa e equivalentes de caixa,Ativo,111010100100001,nan,NaN,Resultado


In [39]:
df_nulos = pd.DataFrame({
    "coluna": df_razao_final.columns,
    "linhas_totais": len(df_razao_final),
    "nulos_qtd": df_razao_final.isna().sum(),
    "nulos_pct": (df_razao_final.isna().mean() * 100).round(2)
})

df_nulos


,coluna,linhas_totais,nulos_qtd,nulos_pct
Conta,Conta,215450,0,0.00
CONTA,CONTA,215450,0,0.00
Descrição,Descrição,215450,0,0.00
DATA,DATA,215450,0,0.00
CONTRAPARTIDA,CONTRAPARTIDA,215450,0,0.00
DEBITO,DEBITO,215450,113627,52.74
CREDITO,CREDITO,215450,113627,52.74
MOVIMENTO,MOVIMENTO,215450,113627,52.74
HISTORICO,HISTORICO,215450,0,0.00
GRUPO_CONTA,GRUPO_CONTA,215450,0,0.00


In [40]:
print("CREDITO preenchido:", df_razao_final["CREDITO"].notna().sum())
print("CREDITO nulo:", df_razao_final["CREDITO"].isna().sum())

print("\nExemplos com CREDITO preenchido:")
print(df_razao_final.loc[df_razao_final["CREDITO"].notna(), ["MOVIMENTO", "DEBITO", "CREDITO"]].head(10))

print("\nLinhas sem DEBITO e sem CREDITO:")
print(df_razao_final.loc[
    df_razao_final["DEBITO"].isna() & df_razao_final["CREDITO"].isna(),
    ["MOVIMENTO", "DEBITO", "CREDITO"]
].head(10))

CREDITO preenchido: 101823
CREDITO nulo: 113627

Exemplos com CREDITO preenchido:
    MOVIMENTO  DEBITO  CREDITO
0     -8000.0     0.0  -8000.0
2      8000.0  8000.0      0.0
4     -8000.0     0.0  -8000.0
6      8000.0  8000.0      0.0
8     -8000.0     0.0  -8000.0
10     8000.0  8000.0      0.0
12    -8000.0     0.0  -8000.0
14     8000.0  8000.0      0.0
16        0.2     0.2      0.0
18       -0.2     0.0     -0.2

Linhas sem DEBITO e sem CREDITO:
    MOVIMENTO  DEBITO  CREDITO
1         NaN     NaN      NaN
3         NaN     NaN      NaN
5         NaN     NaN      NaN
7         NaN     NaN      NaN
9         NaN     NaN      NaN
11        NaN     NaN      NaN
13        NaN     NaN      NaN
15        NaN     NaN      NaN
17        NaN     NaN      NaN
19        NaN     NaN      NaN


In [41]:
print(df_razao_final.columns.tolist())

['Conta', 'CONTA', 'Descrição', 'DATA', 'CONTRAPARTIDA', 'DEBITO', 'CREDITO', 'MOVIMENTO', 'HISTORICO', 'GRUPO_CONTA', 'CLASS_CONTA', 'CONTA_CHAVE', 'CONTRAPARTIDA_CHAVE', 'GRUPO_CTP', 'CLASS_CTP']


## 📑 Reordenação e Tratamento de Data

- Reordenação das colunas antes do processamento
- Padronização dos nomes das colunas
- Conversão da coluna `DATA` para formato `dd/mm/yyyy`
- Tratamento de valores inválidos como nulos

Objetivo: garantir organização estrutural e consistência temporal do dataset antes das próximas etapas do pipeline.

In [42]:
# ✅ PASSO 1 — Reordenar ANTES de qualquer processamento
nova_ordem = ['Conta', 'CONTA', 'Descrição', 'DATA', 'CONTRAPARTIDA',
 'DEBITO', 'CREDITO', 'MOVIMENTO', 'HISTORICO', 'GRUPO_CONTA', 'CLASS_CONTA',
  'CONTA_CHAVE', 'CONTRAPARTIDA_CHAVE', 'GRUPO_CTP', 'CLASS_CTP']


df_razao_FINAL2= df_razao_final[nova_ordem].copy()

# ✅ PASSO 2 — Agora sim processar
df = df_razao_FINAL2.copy()
df.columns = df.columns.str.strip()

# DATA
import datetime

if "DATA" in df.columns:
    def converter_data(x):
        # Já é datetime nativo ou Timestamp
        if isinstance(x, (datetime.datetime, pd.Timestamp)):
            return pd.Timestamp(x)
        # Nulo real
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return pd.NaT
        # String → tenta converter, descarta inválidos ("Conta:", etc.)
        if isinstance(x, str):
            x_strip = x.strip()
            if x_strip == "":
                return pd.NaT
            try:
                return pd.Timestamp(x_strip)
            except Exception:
                return pd.NaT  # ← "Conta:" vira vazio
        return pd.NaT  # qualquer outro tipo desconhecido

    df["DATA"] = df["DATA"].apply(converter_data)
    df["DATA"] = df["DATA"].dt.strftime("%d/%m/%Y")
    df["DATA"] = df["DATA"].fillna("")

# ✅ Verificação rápida
print(df["DATA"].head(5))
print("Vazios:", (df["DATA"] == "").sum())
print("Válidos:", df["DATA"].str.match(r"^\d{2}/\d{2}/\d{4}$", na=False).sum())

df_razao_FINAL2.head(1)

0    02/01/2025
1    02/01/2025
2    03/01/2025
3    03/01/2025
4    07/01/2025
Name: DATA, dtype: object
Vazios: 0
Válidos: 215450


,Conta,CONTA,Descrição,DATA,CONTRAPARTIDA,DEBITO,CREDITO,MOVIMENTO,HISTORICO,GRUPO_CONTA,CLASS_CONTA,CONTA_CHAVE,CONTRAPARTIDA_CHAVE,GRUPO_CTP,CLASS_CTP
0,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-02,nan,0.0,-8000.0,-8000.0,TRANF BANCARIA ORIGEM CX3 0000 00000000,Caixa e equivalentes de caixa,Ativo,111010100100001,nan,NaN,Resultado


## 📊 Diagnóstico da Coluna DATA

Execução de análise detalhada da coluna `DATA`, incluindo:

- Tipo da coluna
- Total de registros
- Amostra inicial de valores
- Valores únicos
- Contagem de datas válidas no padrão `dd/mm/aaaa`
- Quantidade de vazios e valores nulos
- Identificação de valores inválidos

Objetivo: validar a integridade temporal do dataset e identificar possíveis inconsistências antes de avançar no pipeline.

In [43]:
print("📊 DIAGNÓSTICO COMPLETO DA COLUNA DATA")
print("="*60)

col = df["DATA"]  # ← já usa o df processado

print("Tipo da coluna:", col.dtype)
print("Total de linhas:", len(col))

print("\n🔎 Primeiros 10 valores:")
print(col.head(10))

print("\n🔎 Valores únicos (até 10):")
print(col.unique()[:10])

print("\n🔎 Contagem de datas válidas (dd/mm/aaaa):")
validas = col.str.match(r"^\d{2}/\d{2}/\d{4}$", na=False).sum()
print(validas)

print("\n🔎 Contagem de datas VAZIAS (''):")
print((col == "").sum())

print("\n🔎 Contagem de NaN/NaT:")
print(col.isna().sum())

print("\n🔎 Valores inválidos (nem data válida nem vazio):")
invalidos = col[~col.str.match(r"^\d{2}/\d{2}/\d{4}$", na=False) & (col != "") & col.notna()]
print(f"Total: {len(invalidos)}")
if len(invalidos) > 0:
    print(invalidos.unique()[:10])

📊 DIAGNÓSTICO COMPLETO DA COLUNA DATA
Tipo da coluna: object
Total de linhas: 215450

🔎 Primeiros 10 valores:
0    02/01/2025
1    02/01/2025
2    03/01/2025
3    03/01/2025
4    07/01/2025
5    07/01/2025
6    10/01/2025
7    10/01/2025
8    14/01/2025
9    14/01/2025
Name: DATA, dtype: object

🔎 Valores únicos (até 10):
['02/01/2025' '03/01/2025' '07/01/2025' '10/01/2025' '14/01/2025'
 '17/01/2025' '21/01/2025' '24/01/2025' '27/01/2025' '28/01/2025']

🔎 Contagem de datas válidas (dd/mm/aaaa):
215450

🔎 Contagem de datas VAZIAS (''):
0

🔎 Contagem de NaN/NaT:
0

🔎 Valores inválidos (nem data válida nem vazio):
Total: 0


## 🔎 Identificação de Datas Fora do Padrão

Filtragem de registros cuja coluna `DATA` não corresponde ao padrão esperado.

Objetivo:
- Detectar valores inconsistentes
- Identificar possíveis erros de conversão
- Isolar registros problemáticos para correção ou exclusão

Visualização das primeiras ocorrências inválidas para análise.

In [44]:
valores_invalidos = df[~df["DATA"].astype(str).str.contains(r"\d{4}-\d{2}-\d{2}", na=False)]

print("Linhas problemáticas:")
print(valores_invalidos["DATA"].head(10))


Linhas problemáticas:
0    02/01/2025
1    02/01/2025
2    03/01/2025
3    03/01/2025
4    07/01/2025
5    07/01/2025
6    10/01/2025
7    10/01/2025
8    14/01/2025
9    14/01/2025
Name: DATA, dtype: object


In [45]:
print(df_razao_FINAL2.columns.tolist())

['Conta', 'CONTA', 'Descrição', 'DATA', 'CONTRAPARTIDA', 'DEBITO', 'CREDITO', 'MOVIMENTO', 'HISTORICO', 'GRUPO_CONTA', 'CLASS_CONTA', 'CONTA_CHAVE', 'CONTRAPARTIDA_CHAVE', 'GRUPO_CTP', 'CLASS_CTP']


## 💾 Consolidação e Exportação – Camada Silver

### 🔹 Reordenação Estrutural
Definição da ordem padrão das colunas antes de qualquer processamento, garantindo consistência estrutural do dataset.

### 🔹 Tratamentos Aplicados

- Conversão segura da coluna `DATA` para o formato `dd/mm/aaaa`
- Padronização das colunas `CONTA` e `CONTA_CONTRAPARTIDA` (remoção de notação científica e ajustes de string)
- Trim em colunas textuais
- Padronização de valores monetários (`MOVIMENTO`, `DEBITO`, `CREDITO`) com arredondamento em 2 casas decimais

### 🔹 Exportação

- Geração do arquivo `.csv` na camada **Silver**
- Separador `;`
- Decimal `,`
- Encoding `utf-8`
- Quebra de linha padrão `\n`

### 🔹 Diagnóstico Pós-Exportação

- Total de linhas e colunas
- Contagem de datas válidas no padrão `dd/mm/aaaa`
- Contagem de datas vazias

**Objetivo:** consolidar e persistir a base tratada, garantindo integridade estrutural, consistência temporal e padronização numérica antes de etapas analíticas posteriores.

In [46]:
print(df_razao_FINAL2.columns.tolist())

['Conta', 'CONTA', 'Descrição', 'DATA', 'CONTRAPARTIDA', 'DEBITO', 'CREDITO', 'MOVIMENTO', 'HISTORICO', 'GRUPO_CONTA', 'CLASS_CONTA', 'CONTA_CHAVE', 'CONTRAPARTIDA_CHAVE', 'GRUPO_CTP', 'CLASS_CTP']


In [47]:
import os
import datetime
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# ✅ PASSO 1 — DEFINIR ORDEM FINAL DAS COLUNAS
# ─────────────────────────────────────────────
nova_ordem =['Conta', 'CONTA', 'Descrição', 'DATA', 'CONTRAPARTIDA', 
'DEBITO', 'CREDITO', 'MOVIMENTO', 'HISTORICO', 'GRUPO_CONTA', 'CLASS_CONTA',
 'CONTA_CHAVE', 'CONTRAPARTIDA_CHAVE', 'GRUPO_CTP', 'CLASS_CTP']


# Validação de colunas obrigatórias
colunas_faltando = [col for col in nova_ordem if col not in df_razao_FINAL2.columns]
if colunas_faltando:
    raise ValueError(f"❌ Colunas ausentes em df_razao_FINAL2: {colunas_faltando}")

# Reordena e cria cópia segura
df_razao_FINAL3 = df_razao_FINAL2[nova_ordem].copy()

# ─────────────────────────────────────────────
# 📁 CONFIGURAÇÃO DE SAÍDA
# ─────────────────────────────────────────────
pasta_saida = r"C:\AZURE_DEVOS_REPOS\RSM_DATALAKE_JOURNAL_ENTRIES\033-DTLK-Data Analytics_Viação Normandy do Triângulo Ltda_31.12.2025\silver"
os.makedirs(pasta_saida, exist_ok=True)

saida_csv = os.path.join(pasta_saida, "razao_NORMANDY_silver.csv")

# ─────────────────────────────────────────────
# 📋 CÓPIA DO DATAFRAME
# ─────────────────────────────────────────────
df = df_razao_FINAL3.copy()
df.columns = df.columns.str.strip()

# ─────────────────────────────────────────────
# 🔹 FUNÇÃO — CONVERSÃO SEGURA DE DATA
# ─────────────────────────────────────────────
def converter_data_segura(valor):
    if pd.isna(valor):
        return pd.NaT

    # Se já for datetime
    if isinstance(valor, (pd.Timestamp, datetime.datetime, datetime.date)):
        return pd.to_datetime(valor, errors="coerce")

    # Se for número serial do Excel
    if isinstance(valor, (int, float, np.integer, np.floating)):
        try:
            return pd.to_datetime("1899-12-30") + pd.to_timedelta(valor, unit="D")
        except:
            return pd.NaT

    # Se for texto
    if isinstance(valor, str):
        valor = valor.strip()
        if valor == "":
            return pd.NaT

        # Formato BR
        data = pd.to_datetime(valor, format="%d/%m/%Y", errors="coerce")
        if pd.notna(data):
            return data

        # Formato ISO
        data = pd.to_datetime(valor, format="%Y-%m-%d", errors="coerce")
        if pd.notna(data):
            return data

        # Tentativa final controlada
        return pd.to_datetime(valor, dayfirst=True, errors="coerce")

    return pd.NaT

# ─────────────────────────────────────────────
# 🔹 FUNÇÃO — CONVERSÃO SEGURA DE NÚMEROS
#    Não corrompe número que já está correto:
#    Ex.: 292488.05 continua 292488.05
# ─────────────────────────────────────────────
def converter_numero_seguro(valor):
    if pd.isna(valor):
        return np.nan

    # Se já for número real, mantém
    if isinstance(valor, (int, float, np.integer, np.floating)):
        return float(valor)

    s = str(valor).strip()

    if s == "" or s.lower() in ["nan", "none", "nat"]:
        return np.nan

    # Caso tenha ponto e vírgula
    if "." in s and "," in s:
        # Ex.: 1.234,56  -> BR
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        # Ex.: 1,234.56 -> EN
        else:
            s = s.replace(",", "")

    # Caso só tenha vírgula: assume decimal BR
    elif "," in s:
        s = s.replace(".", "").replace(",", ".")

    # Caso só tenha ponto: já pode estar correto, não remove nada

    return pd.to_numeric(s, errors="coerce")

# ─────────────────────────────────────────────
# 🔹 DATA — CONVERSÃO E FORMATAÇÃO FINAL
# ─────────────────────────────────────────────
datas_validas = 0
datas_invalidas = 0

if "DATA" in df.columns:
    df["DATA"] = df["DATA"].apply(converter_data_segura)

    datas_validas = df["DATA"].notna().sum()
    datas_invalidas = df["DATA"].isna().sum()

    df["DATA"] = df["DATA"].dt.strftime("%d/%m/%Y")
    df["DATA"] = df["DATA"].fillna("")

# ─────────────────────────────────────────────
# 🔹 CONTA E CONTRAPARTIDA — SEM NOTAÇÃO CIENTÍFICA
# ─────────────────────────────────────────────
for col in ["CONTA", "CONTRAPARTIDA"]:
    if col in df.columns:
        df[col] = df[col].apply(converter_numero_seguro)

        df[col] = np.where(
            pd.isna(df[col]),
            "",
            df[col].astype("Int64").astype(str)
        )

        # remove "<NA>" gerado pelo Int64
        df[col] = df[col].replace("<NA>", "").astype(str).str.strip()

# ─────────────────────────────────────────────
# 🔹 TRIM EM COLUNAS DE TEXTO (EXCETO DATA)
# ─────────────────────────────────────────────
for col in df.select_dtypes(include=["object"]).columns:
    if col != "DATA":
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace("nan", "").replace("None", "")

# ─────────────────────────────────────────────
# 🔹 VALORES MONETÁRIOS — CONVERSÃO SEGURA
#    Não remove ponto decimal quando o valor já está correto
# ─────────────────────────────────────────────
for col in ["MOVIMENTO", "DEBITO", "CREDITO", "Saldo"]:
    if col in df.columns:
        df[col] = df[col].apply(converter_numero_seguro)
        df[col] = df[col].round(2)

# ─────────────────────────────────────────────
# 🔹 DIAGNÓSTICOS ANTES DA EXPORTAÇÃO
# ─────────────────────────────────────────────
credito_preenchido = df["CREDITO"].notna().sum() if "CREDITO" in df.columns else 0
credito_nulo = df["CREDITO"].isna().sum() if "CREDITO" in df.columns else 0
debito_preenchido = df["DEBITO"].notna().sum() if "DEBITO" in df.columns else 0
debito_nulo = df["DEBITO"].isna().sum() if "DEBITO" in df.columns else 0
mov_zero = (df["MOVIMENTO"] == 0).sum() if "MOVIMENTO" in df.columns else 0

# ─────────────────────────────────────────────
# 🔥 EXPORTAÇÃO CSV
# ─────────────────────────────────────────────
df.to_csv(
    saida_csv,
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    lineterminator="\n"
)

# ─────────────────────────────────────────────
# ✅ SAÍDA FINAL
# ─────────────────────────────────────────────
print("✅ CSV exportado com sucesso!")
print(f"📁 Caminho: {saida_csv}")
print(f"📊 Linhas: {df.shape[0]}")
print(f"📊 Colunas: {df.shape[1]}")
print(f"📅 Datas válidas (dd/mm/aaaa): {datas_validas}")
print(f"⚠️ Datas inválidas/vazias: {datas_invalidas}")
print(f"💳 DEBITO preenchido: {debito_preenchido}")
print(f"💳 DEBITO nulo: {debito_nulo}")
print(f"💰 CREDITO preenchido: {credito_preenchido}")
print(f"💰 CREDITO nulo: {credito_nulo}")
print(f"➖ MOVIMENTO = 0: {mov_zero}")

✅ CSV exportado com sucesso!
📁 Caminho: C:\AZURE_DEVOS_REPOS\RSM_DATALAKE_JOURNAL_ENTRIES\033-DTLK-Data Analytics_Viação Normandy do Triângulo Ltda_31.12.2025\silver\razao_NORMANDY_silver.csv
📊 Linhas: 215450
📊 Colunas: 15
📅 Datas válidas (dd/mm/aaaa): 215450
⚠️ Datas inválidas/vazias: 0
💳 DEBITO preenchido: 101823
💳 DEBITO nulo: 113627
💰 CREDITO preenchido: 101823
💰 CREDITO nulo: 113627
➖ MOVIMENTO = 0: 0


In [48]:
df_razao_FINAL3.head(1)

,Conta,CONTA,Descrição,DATA,CONTRAPARTIDA,DEBITO,CREDITO,MOVIMENTO,HISTORICO,GRUPO_CONTA,CLASS_CONTA,CONTA_CHAVE,CONTRAPARTIDA_CHAVE,GRUPO_CTP,CLASS_CTP
0,1.1.1,111010100100001,NUMERARIO EM CAIXA,2025-01-02,nan,0.0,-8000.0,-8000.0,TRANF BANCARIA ORIGEM CX3 0000 00000000,Caixa e equivalentes de caixa,Ativo,111010100100001,nan,NaN,Resultado


## 📥 Validação da Camada Silver

### 🔹 Leitura do CSV

Importação do arquivo Silver utilizando:

- Separador `;`
- Encoding `utf-8`
- Tipagem inicial como `string` para controle manual de conversão

---

### 🔹 Tratamento Aplicado

**DATA**
- Conversão de `dd/mm/aaaa` para datetime
- Remoção de datas inválidas
- Reaplicação do formato `dd/mm/aaaa`

**Campos Numéricos**
- Substituição de vírgula por ponto
- Conversão para tipo numérico
- Tratamento de valores inválidos como `NaN`

**Campos Textuais**
- Padronização com `.str.strip()` para evitar inconsistências

---

### 🔹 Diagnóstico Final

- Total de linhas e colunas
- Quantidade de datas válidas
- Contagem de valores nulos em campos monetários

**Objetivo:** garantir integridade estrutural, consistência temporal e correção numérica antes de análises ou aplicação de testes contábeis.

In [49]:
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# 📥 LEITURA DO CSV SILVER
# ─────────────────────────────────────────────
caminho_csv = r"C:\AZURE_DEVOS_REPOS\RSM_DATALAKE_JOURNAL_ENTRIES\033-DTLK-Data Analytics_Viação Normandy do Triângulo Ltda_31.12.2025\silver\razao_NORMANDY_silver.csv"

df_validacao = pd.read_csv(
    caminho_csv,
    sep=";",
    encoding="utf-8",
    dtype=str
)

# ─────────────────────────────────────────────
# 🔹 DATA — dd/mm/aaaa → datetime → mantém barras
# ─────────────────────────────────────────────
df_validacao["DATA"] = pd.to_datetime(
    df_validacao["DATA"].str.strip(),
    format="%d/%m/%Y",
    errors="coerce"
)

# Remove linhas sem data válida
df_validacao = df_validacao[df_validacao["DATA"].notna()].reset_index(drop=True)

# ✅ Formata de volta para string dd/mm/aaaa com barras
df_validacao["DATA"] = df_validacao["DATA"].dt.strftime("%d/%m/%Y")

# ─────────────────────────────────────────────
# 🔹 NUMÉRICOS — vírgula → ponto e converte
# ─────────────────────────────────────────────
for col in ["MOVIMENTO", "DEBITO", "CREDITO", "Saldo", "Lanç"]:
    if col in df_validacao.columns:
        df_validacao[col] = (
            df_validacao[col]
            .str.strip()
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
        )
        df_validacao[col] = pd.to_numeric(df_validacao[col], errors="coerce")

# ─────────────────────────────────────────────
# 🔹 TEXTOS
# ─────────────────────────────────────────────
for col in ["CLASS_CTP", "CLASS_CONTA", "GRUPO_CONTA", "GRUPO_CTP"]:
    if col in df_validacao.columns:
        df_validacao[col] = df_validacao[col].astype(str).str.strip()

# ─────────────────────────────────────────────
# ✅ DIAGNÓSTICO
# ─────────────────────────────────────────────
pattern = r"^\d{2}/\d{2}/\d{4}$"
datas_validas = df_validacao["DATA"].str.match(pattern, na=False).sum()

print(f"📊 Linhas:  {df_validacao.shape[0]}")
print(f"📊 Colunas: {df_validacao.shape[1]}")
print(f"📅 Datas válidas (dd/mm/aaaa): {datas_validas}")
print(f"\n💰 MOVIMENTO — nulos: {df_validacao['MOVIMENTO'].isna().sum()}")
print(f"💰 DEBITO    — nulos: {df_validacao['DEBITO'].isna().sum()}")
print(f"💰 CREDITO   — nulos: {df_validacao['CREDITO'].isna().sum()}")

df_validacao.head(3)

📊 Linhas:  215450
📊 Colunas: 15
📅 Datas válidas (dd/mm/aaaa): 215450

💰 MOVIMENTO — nulos: 113627
💰 DEBITO    — nulos: 113627
💰 CREDITO   — nulos: 113627


,Conta,CONTA,Descrição,DATA,CONTRAPARTIDA,DEBITO,CREDITO,MOVIMENTO,HISTORICO,GRUPO_CONTA,CLASS_CONTA,CONTA_CHAVE,CONTRAPARTIDA_CHAVE,GRUPO_CTP,CLASS_CTP
0,1.1.1,111010100100001,NUMERARIO EM CAIXA,02/01/2025,NaN,0.0,-8000.0,-8000.0,TRANF BANCARIA ORIGEM CX3 0000 00000000,Caixa e equivalentes de caixa,Ativo,111010100100001,NaN,nan,Resultado
1,1.1.1,111010100100001,NUMERARIO EM CAIXA,02/01/2025,NaN,NaN,NaN,NaN,00,Caixa e equivalentes de caixa,Ativo,111010100100001,NaN,nan,Resultado
2,1.1.1,111010100100001,NUMERARIO EM CAIXA,03/01/2025,NaN,8000.0,0.0,8000.0,TRANF BANCARIA DESTINO CX3 0000 0000000,Caixa e equivalentes de caixa,Ativo,111010100100001,NaN,nan,Resultado


## 📋 Análise de Domínios – Colunas Classificatórias

Listagem dos valores únicos das colunas:

- `GRUPO_CONTA`
- `GRUPO_CTP`
- `CLASS_CONTA`
- `CLASS_CTP`

Procedimento:
- Conversão para string
- Extração de valores distintos (`unique()`)
- Exibição ordenada

Objetivo: identificar o domínio de classificações existentes e validar consistência nas categorias contábeis antes da aplicação de regras ou testes.

In [50]:
colunas_dominios = [
    "GRUPO_CONTA",
    "GRUPO_CTP",
    "CLASS_CONTA",
    "CLASS_CTP"
]

for col in colunas_dominios:
    print(f"\n==================== {col} ====================")
    
    valores = (
        df_validacao[col]
        .astype(str)
        .unique()
    )
    
    for v in sorted(valores):
        print(f"'{v}'")



==================== GRUPO_CONTA ====================
'Adiantamentos diversos'
'Caixa e equivalentes de caixa'
'Contas a receber'
'Custos com arrendamento mercantil e locações'
'Custos com depreciação e amortização'
'Custos com pessoal'
'Custos com veículos'
'Deduções da receita bruta'
'Depósitos judiciais'
'Despesas com depreciação e amortização'
'Despesas com pessoal'
'Despesas comerciais'
'Despesas financeiras'
'Despesas gerais e administrativas'
'Estoques'
'Fornecedores - CP'
'Imobilizado'
'Imposto de renda e contribuição social - diferido'
'Impostos e contribuições a recolher'
'Impostos e contribuições a recuperar - CP'
'Intangível'
'Investimentos'
'Lucros / Prejuízos acumulados'
'Obrigações sociais e trabalhistas'
'Outras contas a pagar - CP'
'Outras despesas'
'Outras receitas'
'Outros créditos - CP'
'Outros custos operacionais'
'Provisão para contingências - LP'
'Provisão sobre a folha de pagamento'
'Receita bruta'
'Receitas financeiras'
'Transações entre partes relacionadas'



# TESTE_7_7 GRUPO_CONTA >>> GRUPO_CAIXA_TESTE_7_7
'Caixa e equivalentes de caixa'
# TESTE_7_7 CLASS_CTP >>> CLASS_RESULTADO_TESTE_7_7
# ==========================================================
Resultado

PREVIEW



## ⚙️ Configuração Base – Preview de Auditoria

### 🔹 Arquivo de Origem
Definição do caminho do arquivo Razão (camada Silver) e separador CSV utilizado na leitura.

---

### 🔹 Ativação de Testes

Configuração central (`TESTES_CONFIG`) para controle de execução dos testes automatizados de auditoria.

Objetivo:
- Permitir ativação/desativação individual de cada teste
- Facilitar manutenção e versionamento das regras

---

### 🔹 Datas de Referência 2025

Listas auxiliares contendo:

- Feriados oficiais
- Finais de semana pré-calculados

Objetivo:
- Suporte a testes que validam movimentações em dias não úteis
- Padronização entre execução Pandas e Spark

---

### 🔹 Listas Textuais para Testes

Definição de palavras-chave e nomes específicos utilizados em testes de classificação e detecção de padrões:

- Ajustes e estornos
- Variação cambial
- Pessoas específicas (TESTE_1_8)
- Categorias contábeis por grupo

Objetivo:
- Centralizar regras de negócio
- Evitar hardcode distribuído ao longo do script
- Facilitar auditoria e revisão técnica

---

### 🎯 Finalidade Geral

Estabelecer um bloco único de configuração para:

- Controle de execução dos testes
- Parametrização de regras contábeis
- Padronização da auditoria automatizada
- Manutenção escalável do framework

In [3]:
# ========================================
# CONFIGURAÇÕES BASE DO PREVIEW
# ========================================

# 🔹 Caminho do seu arquivo razão
ARQ_RAZAO = r"C:\AZURE_DEVOS_REPOS\RSM_DATALAKE_JOURNAL_ENTRIES\034-DTLK-Data analytics_GRAHAM_31\silver\razao 11e 12.csv"

# ==========================================================
# CONSTANTES PARA TESTES DE CLASSIFICAÇÃO
# ==========================================================

# ==========================================================
# 🔹 CONFIGURAÇÃO BASE
# ==========================================================

CSV_SEPARATOR = ";"

TESTES_CONFIG = {
    'TESTE_1_1': False,
    'TESTE_1_2': False,
    'TESTE_1_3': True,
    'TESTE_1_4': True,
    'TESTE_1_5': True,
    'TESTE_1_6': False,
    'TESTE_1_7': True,
    'TESTE_1_8': True,
    'TESTE_2_1': False,
    'TESTE_3_1': False,
    'TESTE_4_1': False,
    'TESTE_5_1': False,
    'TESTE_6_1': False,
    'TESTE_7_1': False,
    'TESTE_7_2': False,
    'TESTE_7_3': False,
    'TESTE_7_4': False,
    'TESTE_7_5': False,
    'TESTE_7_6': False,
    'TESTE_7_7': False,
    'TESTE_7_9': False
}

# ==========================================================
# 🔹 FERIADOS 2025
# ==========================================================

# 🔹 Dataset oficial de feriados
FERIADOS = [ 
"01/01/2025",
"20/01/2025",
"13/02/2025",
"29/03/2025",
"21/04/2025",
"23/04/2025",
"01/05/2025",
"30/05/2025",
"07/09/2025",
"12/10/2025",
"02/11/2025",
"15/11/2025",
"20/11/2025",
"25/12/2025"
]

# 🔹 Dataset oficial de finais de semana (pré-calculado igual Spark)
FINAIS_DE_SEMANA = [
    "04/01/2025","05/01/2025",
    "11/01/2025","12/01/2025",
    "18/01/2025","19/01/2025",
    "25/01/2025","26/01/2025",
    "01/02/2025","02/02/2025",
    "08/02/2025","09/02/2025",
    "15/02/2025","16/02/2025",
    "22/02/2025","23/02/2025",
    "01/03/2025","02/03/2025",
    "08/03/2025","09/03/2025",
    "15/03/2025","16/03/2025",
    "22/03/2025","23/03/2025",
    "29/03/2025","30/03/2025",
    "05/04/2025","06/04/2025",
    "12/04/2025","13/04/2025",
    "19/04/2025","20/04/2025",
    "26/04/2025","27/04/2025",
    "03/05/2025","04/05/2025",
    "10/05/2025","11/05/2025",
    "17/05/2025","18/05/2025",
    "24/05/2025","25/05/2025",
    "31/05/2025","01/06/2025",
    "07/06/2025","08/06/2025",
    "14/06/2025","15/06/2025",
    "21/06/2025","22/06/2025",
    "28/06/2025","29/06/2025",
    "05/07/2025","06/07/2025",
    "12/07/2025","13/07/2025",
    "19/07/2025","20/07/2025",
    "26/07/2025","27/07/2025",
    "02/08/2025","03/08/2025",
    "09/08/2025","10/08/2025",
    "16/08/2025","17/08/2025",
    "23/08/2025","24/08/2025",
    "30/08/2025","31/08/2025",
    "06/09/2025","13/09/2025","14/09/2025",
    "20/09/2025","21/09/2025",
    "27/09/2025","28/09/2025",
    "04/10/2025","05/10/2025",
    "11/10/2025","18/10/2025","19/10/2025",
    "25/10/2025","26/10/2025",
    "01/11/2025","08/11/2025","09/11/2025",
    "16/11/2025",
    "22/11/2025","23/11/2025",
    "29/11/2025","30/11/2025",
    "06/12/2025","07/12/2025",
    "13/12/2025","14/12/2025",
    "20/12/2025","21/12/2025",
    "27/12/2025","28/12/2025",
]

# ==========================================================
# 🔹 LISTAS TEXTUAIS (evita NameError)
# ==========================================================

PALAVRAS_TESTE_1_4 = [
    'segundo', 'acordo', 'ordem', 'mail', 
    'conforme', 'controller', 'pedido'
]

# Palavras-chave para TESTE_1_5 (ajuste/acerto)
PALAVRAS_TESTE_1_5 = [
    'ajuste', 'acerto'
]

# Palavras-chave para TESTE_1_6 (reversão/estorno)
PALAVRAS_TESTE_1_6 = [
    'reversao', 'reversão', 'estorno'
]

# Palavras-chave para TESTE_1_7 (outros/diversas)
PALAVRAS_TESTE_1_7 = [
    'outros', 'outras', 'desconhecidos', 'diversos'
]

# Palavras-chave para TESTE_7_6 (variação cambial)
PALAVRAS_TESTE_7_6 = [
    'variacao cambial', 'variação cambial', 'cambial'
]
 

NOMES_TESTE_1_8 = [
"GABRIELA POIATTO TEIXEIRA",
"GABRIELA",
"POIATTO",
"TEIXEIRA",

"TATIANE SILVA MATHEW",
"TATIANE",
"SILVA",
"MATHEW",

"DENNY JOSEPH",
"DENNY",
"JOSEPH",

"WANDERLEI ANTONIO",
"WANDERLEI",
"ANTONIO",

"TIAGO ALEXANDRE",
"TIAGO",
"ALEXANDRE"
]


NOMES_TESTE_7_9 = ['']

# ==========================================================
# TESTE_2_1 >> CLASS_CTP
# ==========================================================
CLASS_RESULTADO_TESTE_2_1 = ['Resultado']

# ==========================================================
# TESTE_6_1 >> GRUPO_CONTA
# ==========================================================

GRUPOS_RECEITA_TESTE_6_1 = [
    'Receita líquida'
    'Receitas financeiras'
]

# ==========================================================
# TESTE_7_1 >> CLASS_CONTA
# ==========================================================

CLASS_CONTA_TESTE_7_1 = ['Resultado']


# ==========================================================
# TESTE_7_2 >> GRUPO_CTP >>> GRUPOS_RECEITA_TESTE_7_2
# TESTE_7_2 >> GRUPO_CONTA >>> GRUPO_IMOBILIZADO_TESTE_7_2
# ==========================================================

GRUPOS_RECEITA_TESTE_7_2 = [
    'Outras receitas',
    'Receitas financeiras',
    'Receita bruta'
]

GRUPO_IMOBILIZADO_TESTE_7_2 = ['Imobilizado']

# ==========================================================
# TESTE_7_4 GRUPO_CONTA >>> GRUPO_IMOBILIZADO_TESTE_7_4
# TESTE_7_4 GRUPO_CONTA >>> GRUPOS_EXCLUIR_TESTE_7_4
# ==========================================================

GRUPO_IMOBILIZADO_TESTE_7_4 = ['Imobilizado']

GRUPOS_EXCLUIR_TESTE_7_4 = [
    'Caixa e equivalentes de caixa',  
    'Fornecedores'
]

# ==========================================================
# TESTE_7_5  GRUPO_CONTA >>> GRUPOS_RECEITA_TESTE_7_5
# TESTE_7_5  GRUPO_CTP >>> GRUPOS_CONTAS_RECEBER_TESTE_7_5
# ==========================================================

GRUPOS_RECEITA_TESTE_7_5 = [
    'receita líquida',
    'receitas financeiras'
]

GRUPOS_CONTAS_RECEBER_TESTE_7_5 = ['contas a receber']

# ==========================================================
# TESTE_7_6  GRUPO_CONTA >>> GRUPOS_EXCLUIR_TESTE_7_6
# ==========================================================

GRUPOS_EXCLUIR_TESTE_7_6 = [
    'Emprestimos',
    'Partes relacionadas'
]

# ==========================================================
# TESTE_7_7 GRUPO_CONTA >>> GRUPO_CAIXA_TESTE_7_7
# TESTE_7_7 CLASS_CTP >>> CLASS_RESULTADO_TESTE_7_7
# ==========================================================
 

GRUPO_CAIXA_TESTE_7_7 = ['Caixa e equivalentes de caixa']

CLASS_RESULTADO_TESTE_7_7 = ['Resultado']


## 🔎 Auditoria Contábil – Preview Total Parametrizado (Oficial)

### 🚀 Inicialização do Ambiente

Criação de sessão Spark local para execução do preview:

- Execução em `local[*]`
- Timezone configurado para `America/Sao_Paulo`
- Política de parsing compatível (`LEGACY`)
- Logs reduzidos para `ERROR`

Objetivo: permitir validação controlada das regras antes de execução em ambiente produtivo.

---

### 📥 Carregamento do Arquivo Razão

Leitura do CSV da camada Silver com:

- Header habilitado
- Separador parametrizado
- Inferência automática de schema
- Suporte a multiline

Validação inicial:
- Total de registros
- Total de colunas

---

### 🔧 Normalizações Aplicadas

**Textuais**
- `lower()` + `trim()` para padronização
- Preparação para uso seguro de `.isin()` e `.like()`

**Monetárias**
- Conversão padrão BR → double
  - Remove ponto de milhar
  - Converte vírgula decimal
  - Remove caracteres inválidos

**Datas**
- Criação de `DATA_DATE` com múltiplos formatos suportados
- Base para testes que envolvem calendário

---

### 🪟 Preparação de Janelas Analíticas

Criação de:
- `LINHA_SEQ` (particionamento por CONTA + MOVIMENTO)
- `LINHA_SEQ_TESTE4` (particionamento por HISTORICO + MOVIMENTO + DATA)

Objetivo: suporte a testes de repetição e sequencialidade.

---

### 🔒 Blindagem das Listas de Regras

Normalização prévia das listas utilizadas em:

- `.isin()`
- `.like()`
- Regex de nomes

Objetivo: evitar inconsistência por caixa (upper/lower) e espaços.

---

### 🧪 Aplicação dos Testes Automatizados

Execução condicional conforme `TESTES_CONFIG`, incluindo:

- Testes estruturais (datas nulas, histórico nulo)
- Testes por palavras-chave
- Testes por grupos contábeis
- Testes monetários (valores arredondados)
- Testes por período (mês específico)
- Testes por regras combinadas débito/crédito
- Amostragem aleatória controlada

Cada teste gera coluna própria (`SIM` ou `null`).

---

### 🚩 Flag Final

Criação da coluna `FLAG`:

- `RETIRAR` → nenhum teste sinalizado
- `MANTER` → pelo menos um teste sinalizado

Objetivo: consolidação da decisão automática.

---

### 📊 Resultados

- Resumo geral por FLAG
- Quantidade de ocorrências por teste
- Amostras de registros classificados como:
  - `MANTER`
  - `RETIRAR`

---

### 🎯 Finalidade Geral

Implementar um framework parametrizado de auditoria contábil com:

- Regras configuráveis
- Execução controlada
- Padronização Spark ↔ Pandas
- Estrutura pronta para escalar para ambiente produtivo

In [4]:
# ==========================================================
# 🔎 AUDITORIA CONTÁBIL - PREVIEW TOTAL PARAMETRIZADO (OFICIAL)
# ==========================================================

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DecimalType
from pyspark.sql.window import Window
from datetime import datetime
import re
from unidecode import unidecode

# ==========================================================
# 🚀 CRIAR SPARK LEVE PARA PREVIEW
# ==========================================================

spark = SparkSession.builder \
    .appName("AuditoriaPreviewParametrizada_OFICIAL") \
    .master("local[*]") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.sql.session.timeZone", "America/Sao_Paulo") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Spark iniciado")

# ==========================================================
# 📥 CARREGAR ARQUIVO RAZÃO
# ==========================================================

df = spark.read \
    .option("header", True) \
    .option("sep", CSV_SEPARATOR) \
    .option("inferSchema", True) \
    .option("multiline", "true") \
    .csv(ARQ_RAZAO)

print("📊 Total registros:", df.count())
print("📐 Total colunas :", len(df.columns))

# ==========================================================
# 🔧 FUNÇÕES AUXILIARES
# ==========================================================

def converter_colunas_monetarias_br(df, colunas):
    """
    Converte colunas monetárias no padrão BR para double:
    - remove ponto de milhar
    - troca vírgula por ponto
    - remove lixo
    """
    for c in colunas:
        if c in df.columns:
            df = df.withColumn(c, F.col(c).cast("string"))
            df = df.withColumn(c, F.regexp_replace(F.col(c), r"\.", ""))      # milhar
            df = df.withColumn(c, F.regexp_replace(F.col(c), ",", "."))      # decimal
            df = df.withColumn(c, F.regexp_replace(F.col(c), r"[^0-9\.\-]", ""))
            df = df.withColumn(c, F.when(F.length(F.col(c)) == 0, None).otherwise(F.col(c)))
            df = df.withColumn(c, F.col(c).cast("double"))
    return df

def normalizar_texto_cols(df, colunas):
    for c in colunas:
        if c in df.columns:
            df = df.withColumn(c, F.lower(F.trim(F.col(c).cast("string"))))
    return df

def lower_list(lista):
    return [str(x).strip().lower() for x in lista] if lista else []

def cond_like(coluna, lista):
    """
    Like simples (contém) - lista já deve vir normalizada para lower.
    """
    cond = F.lit(False)
    for p in lista:
        p = str(p).strip().lower()
        if p:
            cond = cond | F.lower(coluna).like(f"%{p}%")
    return cond

def build_regex_nomes(nomes):
    """
    Regex com proteção para nomes curtos:
    - palavra única <= 5 chars => usa \bpalavra\b
    - nome composto => usa match do texto (escapado)
    """
    if not nomes:
        return None

    regex_lista = []
    for nome in nomes:
        nome_norm = unidecode(str(nome).strip().lower())
        partes = nome_norm.split()

        if len(partes) == 1 and len(partes[0]) <= 5:
            palavra = re.escape(partes[0])
            regex_lista.append(rf"\b{palavra}\b")
        else:
            regex_lista.append(re.escape(nome_norm))

    return "|".join(regex_lista) if regex_lista else None

def verificar_colunas_necessarias(df, colunas_necessarias):
    """Verifica se as colunas necessárias existem no DataFrame"""
    colunas_faltantes = [col_name for col_name in colunas_necessarias if col_name not in df.columns]
    
    if colunas_faltantes:
        return False
    return True

# ==========================================================
# 🔄 NORMALIZAÇÃO (IGUAL OFICIAL)
# ==========================================================

# Textuais (lower + trim)
df = normalizar_texto_cols(df, ["GRUPO_CONTA","CLASS_CONTA","CLASS_CTP","GRUPO_CTP","HISTORICO"])

# Monetárias (BR -> double)
df = converter_colunas_monetarias_br(df, ["MOVIMENTO","DEBITO","CREDITO"])

# DATA_DATE (3 formatos como oficial)
if "DATA" in df.columns:
    df = df.withColumn(
        "DATA_DATE",
        F.coalesce(
            F.to_date(F.col("DATA"), "dd/MM/yyyy"),
            F.to_date(F.col("DATA"), "yyyy-MM-dd"),
            F.to_date(F.col("DATA"), "yyyy-MM-dd HH:mm:ss")
        )
    )
else:
    df = df.withColumn("DATA_DATE", F.lit(None).cast("date"))

# ==========================================================
# 🧪 PREPARAÇÃO JANELAS
# ==========================================================

window_seq = Window.partitionBy("CONTA","MOVIMENTO").orderBy(F.col("CONTA").asc())
window_seq_t4 = Window.partitionBy("HISTORICO","MOVIMENTO","DATA").orderBy(F.col("HISTORICO").asc())

df = df.withColumn("LINHA_SEQ", F.row_number().over(window_seq))
df = df.withColumn("LINHA_SEQ_TESTE4", F.row_number().over(window_seq_t4))

# ==========================================================
# 🔒 BLINDAGEM: NORMALIZAR LISTAS PARA .isin()
# ==========================================================

CLASS_RESULTADO_TESTE_2_1 = lower_list(CLASS_RESULTADO_TESTE_2_1)
GRUPOS_RECEITA_TESTE_6_1 = lower_list(GRUPOS_RECEITA_TESTE_6_1)
CLASS_CONTA_TESTE_7_1 = lower_list(CLASS_CONTA_TESTE_7_1)

GRUPOS_RECEITA_TESTE_7_2 = lower_list(GRUPOS_RECEITA_TESTE_7_2)
GRUPO_IMOBILIZADO_TESTE_7_2 = lower_list(GRUPO_IMOBILIZADO_TESTE_7_2)

GRUPO_IMOBILIZADO_TESTE_7_4 = lower_list(GRUPO_IMOBILIZADO_TESTE_7_4)
GRUPOS_EXCLUIR_TESTE_7_4 = lower_list(GRUPOS_EXCLUIR_TESTE_7_4)

GRUPOS_RECEITA_TESTE_7_5 = lower_list(GRUPOS_RECEITA_TESTE_7_5)
GRUPOS_CONTAS_RECEBER_TESTE_7_5 = lower_list(GRUPOS_CONTAS_RECEBER_TESTE_7_5)

GRUPOS_EXCLUIR_TESTE_7_6 = lower_list(GRUPOS_EXCLUIR_TESTE_7_6)

GRUPO_CAIXA_TESTE_7_7 = lower_list(GRUPO_CAIXA_TESTE_7_7)
CLASS_RESULTADO_TESTE_7_7 = lower_list(CLASS_RESULTADO_TESTE_7_7)

PALAVRAS_TESTE_1_4 = lower_list(PALAVRAS_TESTE_1_4)
PALAVRAS_TESTE_1_5 = lower_list(PALAVRAS_TESTE_1_5)
PALAVRAS_TESTE_1_6 = lower_list(PALAVRAS_TESTE_1_6)
PALAVRAS_TESTE_1_7 = lower_list(PALAVRAS_TESTE_1_7)
PALAVRAS_TESTE_7_6 = lower_list(PALAVRAS_TESTE_7_6)

NOMES_TESTE_1_8 = NOMES_TESTE_1_8 or []
NOMES_TESTE_7_9 = NOMES_TESTE_7_9 or []

# ==========================================================
# 🧪 APLICAR TESTES
# ==========================================================

for teste, ativo in TESTES_CONFIG.items():

    if not ativo:
        df = df.withColumn(teste, F.lit(None))
        continue

    # -----------------------------
    # TESTES 1.x
    # -----------------------------
    if teste == "TESTE_1_1":
        df = df.withColumn(teste, F.when(F.col("DATA").isNull(), "SIM"))

    elif teste == "TESTE_1_2":
        todas_datas = (FERIADOS or []) + (FINAIS_DE_SEMANA or [])
        datas_iso = [datetime.strptime(d, "%d/%m/%Y").strftime("%Y-%m-%d") for d in todas_datas]

        df = df.withColumn(
            teste,
            F.when(
                F.date_format(F.col("DATA_DATE"), "yyyy-MM-dd").isin(datas_iso),
                "SIM"
            )
        )

    elif teste == "TESTE_1_3":
        df = df.withColumn(teste, F.when(F.col("HISTORICO").isNull(), "SIM"))

    elif teste == "TESTE_1_4":
        df = df.withColumn(teste, F.when(cond_like(F.col("HISTORICO"), PALAVRAS_TESTE_1_4), "SIM"))

    elif teste == "TESTE_1_5":
        df = df.withColumn(teste, F.when(cond_like(F.col("HISTORICO"), PALAVRAS_TESTE_1_5), "SIM"))

    elif teste == "TESTE_1_6":
        df = df.withColumn(teste, F.when(cond_like(F.col("HISTORICO"), PALAVRAS_TESTE_1_6), "SIM"))

    elif teste == "TESTE_1_7":
        df = df.withColumn(teste, F.when(cond_like(F.col("HISTORICO"), PALAVRAS_TESTE_1_7), "SIM"))

    elif teste == "TESTE_1_8":
        # Versão oficial (regex + fronteira para nomes curtos)
        if NOMES_TESTE_1_8:
            regex_final = build_regex_nomes(NOMES_TESTE_1_8)
            if regex_final:
                df = df.withColumn(
                    "__HIST_NORM",
                    F.lower(F.col("HISTORICO"))
                )
                df = df.withColumn(
                    teste,
                    F.when(F.col("__HIST_NORM").rlike(regex_final), "SIM").otherwise(F.lit(None))
                ).drop("__HIST_NORM")
            else:
                df = df.withColumn(teste, F.lit(None))
        else:
            df = df.withColumn(teste, F.lit(None))

    # -----------------------------
    # TESTE_2_1
    # -----------------------------
    elif teste == "TESTE_2_1":
        df = df.withColumn(
            teste,
            F.when(
                (F.col("LINHA_SEQ") > 1) &
                (F.trim(F.lower(F.col("CLASS_CTP"))).isin(CLASS_RESULTADO_TESTE_2_1)),
                "SIM"
            )
        )

    # -----------------------------
    # TESTE_3_1 (IGUAL OFICIAL)
    # -----------------------------
    elif teste == "TESTE_3_1":
        mov_dec = F.col("MOVIMENTO").cast(DecimalType(18, 2))
        valor_abs = F.abs(mov_dec)

        df = df.withColumn(
            teste,
            F.when(
                (valor_abs.isNotNull()) &
                (valor_abs >= 10) &
                (F.pmod(valor_abs, 1) == 0) &
                (F.pmod(valor_abs, 10) == 0),
                "SIM"
            ).otherwise(F.lit(None))
        )

    # -----------------------------
    # TESTE_4_1
    # -----------------------------
    elif teste == "TESTE_4_1":
        df = df.withColumn(teste, F.when(F.col("LINHA_SEQ_TESTE4") >= 3, "SIM"))

    # -----------------------------
    # TESTE_5_1
    # -----------------------------
    elif teste == "TESTE_5_1":
        df = df.withColumn("__ID_T51", F.monotonically_increasing_id())
        amostra = (
            df.select("__ID_T51")
              .orderBy(F.rand(42))
              .limit(25)
              .withColumn("__FLAG_T51", F.lit(1))
        )
        df = df.join(amostra, on="__ID_T51", how="left")
        df = df.withColumn(teste, F.when(F.col("__FLAG_T51") == 1, "SIM").otherwise(F.lit(None))) \
               .drop("__ID_T51", "__FLAG_T51")

    # -----------------------------
    # TESTE_6_1
    # -----------------------------
    elif teste == "TESTE_6_1":
        df = df.withColumn(
            teste,
            F.when(
                (F.col("MOVIMENTO") > 30000) &
                (F.col("GRUPO_CONTA").isin(GRUPOS_RECEITA_TESTE_6_1)),
                "SIM"
            )
        )

    # -----------------------------
    # TESTE_7_1
    # -----------------------------
    elif teste == "TESTE_7_1":
        df = df.withColumn(
            teste,
            F.when(F.trim(F.lower(F.col("CLASS_CONTA"))).isin(CLASS_CONTA_TESTE_7_1), "SIM")
        )

    # -----------------------------
    # TESTE_7_2
    # -----------------------------
    elif teste == "TESTE_7_2":
        df = df.withColumn(
            teste,
            F.when(
                (F.coalesce(F.col("DEBITO").cast("double"), F.lit(0.0)) != 0) &
                (F.col("GRUPO_CONTA").isin(GRUPO_IMOBILIZADO_TESTE_7_2)) &
                (F.col("GRUPO_CTP").isin(GRUPOS_RECEITA_TESTE_7_2)),
                "SIM"
            )
        )

    # -----------------------------
    # TESTE_7_3 (no preview deixa nulo)
    # -----------------------------
    elif teste == "TESTE_7_3":

        if verificar_colunas_necessarias(df, ["DATA_DATE", "CONTA", "MOVIMENTO"]):

            # 31/12/2024
            df_2024 = df.filter(
                (F.year(F.col("DATA_DATE")) == 2024) &
                (F.month(F.col("DATA_DATE")) == 12) &
                (F.dayofmonth(F.col("DATA_DATE")) == 31)
            ).select("CONTA", "MOVIMENTO").distinct().withColumn("tem_2024", F.lit(True))

            # Join
            df = df.join(df_2024, on=["CONTA", "MOVIMENTO"], how="left")

            # Janeiro/2025
            df = df.withColumn(
                teste,
                F.when(
                    (F.year(F.col("DATA_DATE")) == 2025) &
                    (F.month(F.col("DATA_DATE")) == 1) &
                    (F.col("tem_2024") == True),
                    "SIM"
                ).otherwise(F.lit(None))
            ).drop("tem_2024")

        else:
            df = df.withColumn(teste, F.lit(None))

    # -----------------------------
    # TESTE_7_4 (COALESCE COMO OFICIAL)
    # -----------------------------
    elif teste == "TESTE_7_4":

        deb = F.coalesce(F.col("DEBITO").cast("double"), F.lit(0.0))
        cred = F.coalesce(F.col("CREDITO").cast("double"), F.lit(0.0))

        df = df.withColumn(
            teste,
            F.when(
                (
                    (deb != 0) &
                    (F.col("GRUPO_CONTA").isin(*GRUPO_IMOBILIZADO_TESTE_7_4))
                )
                |
                (
                    (cred != 0) &
                    (~F.col("GRUPO_CONTA").isin(*GRUPOS_EXCLUIR_TESTE_7_4))
                ),
                "SIM"
            ).otherwise(None)
        )
    # -----------------------------
    # TESTE_7_5
    # -----------------------------
    elif teste == "TESTE_7_5":

        credito = F.col("CREDITO").cast("double")

        df = df.withColumn(
            teste,
            F.when(
                (credito.isNotNull()) &              # não pode ser null
                (credito != 0) &                     # pode ser positivo ou negativo
                (F.col("GRUPO_CONTA").isin(GRUPOS_RECEITA_TESTE_7_5)) &
                (~F.trim(F.lower(F.col("GRUPO_CTP"))).eqNullSafe("contas a receber")),
                "SIM"
            )
        )

    # -----------------------------
    # TESTE_7_6
    # -----------------------------
    elif teste == "TESTE_7_6":
        df = df.withColumn(
            teste,
            F.when(
                cond_like(F.col("HISTORICO"), PALAVRAS_TESTE_7_6) &
                (~F.col("GRUPO_CONTA").isin(GRUPOS_EXCLUIR_TESTE_7_6)),
                "SIM"
            )
        )

    # -----------------------------
    # TESTE_7_7 (ALINHADO)
    # -----------------------------
    elif teste == "TESTE_7_7":
        df = df.withColumn(
            teste,
            F.when(
                (F.col("MOVIMENTO") < 0) &
                (F.col("GRUPO_CONTA").isin(GRUPO_CAIXA_TESTE_7_7)) &
                (F.col("CLASS_CTP").isin(CLASS_RESULTADO_TESTE_7_7)) &
                (F.month(F.col("DATA_DATE")).isin([1, 2])),
                "SIM"
            ).otherwise(F.lit(None))
        )

    # -----------------------------
    # TESTE_7_9
    # -----------------------------
    elif teste == "TESTE_7_9":
        if NOMES_TESTE_7_9:
            df = df.withColumn(teste, F.when(cond_like(F.col("HISTORICO"), lower_list(NOMES_TESTE_7_9)), "SIM"))
        else:
            df = df.withColumn(teste, F.lit(None))

    else:
        df = df.withColumn(teste, F.lit(None))

# ==========================================================
# 🚩 FLAG FINAL (IGUAL OFICIAL)
# ==========================================================

condicoes = [F.col(t).isNull() for t, v in TESTES_CONFIG.items() if v]
flag_expr = condicoes[0]
for c in condicoes[1:]:
    flag_expr = flag_expr & c

df = df.withColumn("FLAG", F.when(flag_expr, "RETIRAR").otherwise("MANTER"))

# ==========================================================
# 📊 RESULTADOS OTIMIZADOS
# ==========================================================

print("\n📊 RESUMO GERAL")
df.groupBy("FLAG").count().show()

print("\n📊 RESULTADO POR TESTE (agregado único)")
df.agg(*[
    F.sum(F.when(F.col(t) == "SIM", 1).otherwise(0)).alias(t)
    for t, v in TESTES_CONFIG.items() if v
]).show(truncate=False)

print("\n🔎 AMOSTRA MANTER")
display(df.filter(F.col("FLAG") == "MANTER").limit(20))

print("\n🔎 AMOSTRA RETIRAR")
display(df.filter(F.col("FLAG") == "RETIRAR").limit(20))


✅ Spark iniciado
📊 Total registros: 79353
📐 Total colunas : 26

📊 RESUMO GERAL
+-------+-----+
|   FLAG|count|
+-------+-----+
|RETIRAR|45381|
| MANTER|33972|
+-------+-----+


📊 RESULTADO POR TESTE (agregado único)
+---------+---------+---------+---------+---------+
|TESTE_1_3|TESTE_1_4|TESTE_1_5|TESTE_1_7|TESTE_1_8|
+---------+---------+---------+---------+---------+
|33398    |13       |386      |10       |165      |
+---------+---------+---------+---------+---------+


🔎 AMOSTRA MANTER


DataFrame[Company: int, Document #: bigint, Currency: string, Exchange Rate: string, Document Type: string, Document Date: string, Posting Date: string, DATA: string, d: string, Fiscal Year: int, Fiscal Period: int, Entered By: string, Item #: int, CONTA: int, Profit Center: string, Cost Center: int, Posting Key: int, Amount in Document Currency: string, CREDITO: double, DEBITO: double, MOVIMENTO: double, DR / CR: string, HISTORICO: string, _c23: string, _c24: string, _c25: string, DATA_DATE: date, LINHA_SEQ: int, LINHA_SEQ_TESTE4: int, TESTE_1_1: void, TESTE_1_2: void, TESTE_1_3: string, TESTE_1_4: string, TESTE_1_5: string, TESTE_1_6: void, TESTE_1_7: string, TESTE_1_8: string, TESTE_2_1: void, TESTE_3_1: void, TESTE_4_1: void, TESTE_5_1: void, TESTE_6_1: void, TESTE_7_1: void, TESTE_7_2: void, TESTE_7_3: void, TESTE_7_4: void, TESTE_7_5: void, TESTE_7_6: void, TESTE_7_7: void, TESTE_7_9: void, FLAG: string]


🔎 AMOSTRA RETIRAR


DataFrame[Company: int, Document #: bigint, Currency: string, Exchange Rate: string, Document Type: string, Document Date: string, Posting Date: string, DATA: string, d: string, Fiscal Year: int, Fiscal Period: int, Entered By: string, Item #: int, CONTA: int, Profit Center: string, Cost Center: int, Posting Key: int, Amount in Document Currency: string, CREDITO: double, DEBITO: double, MOVIMENTO: double, DR / CR: string, HISTORICO: string, _c23: string, _c24: string, _c25: string, DATA_DATE: date, LINHA_SEQ: int, LINHA_SEQ_TESTE4: int, TESTE_1_1: void, TESTE_1_2: void, TESTE_1_3: string, TESTE_1_4: string, TESTE_1_5: string, TESTE_1_6: void, TESTE_1_7: string, TESTE_1_8: string, TESTE_2_1: void, TESTE_3_1: void, TESTE_4_1: void, TESTE_5_1: void, TESTE_6_1: void, TESTE_7_1: void, TESTE_7_2: void, TESTE_7_3: void, TESTE_7_4: void, TESTE_7_5: void, TESTE_7_6: void, TESTE_7_7: void, TESTE_7_9: void, FLAG: string]